<a href="https://colab.research.google.com/github/alexklupsch/wur_deep_learning/blob/main/single_label_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single Label Image Classification -- UCM airborne images -- Resnet18 Fine-Tuning

This notebook shows an easy implementation of the fine-tuning of a resnet18 model to classify the 21 classes of the UCM dataset showing airborne Land Use categories.
It follows the implementation of a medium article (https://medium.com/@imabhi1216/fine-tuning-a-pre-trained-resnet-18-model-for-image-classification-on-custom-dataset-with-pytorch-02df12e83c2c)
and adds wandb (weights and biases to the project).


## 1. Data Loading and Preparation

In [ ]:
!pip install lightning

In [ ]:
!pip install transformers

In [ ]:
import os
import zipfile
import random
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor
from PIL import Image
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from lightning.pytorch import LightningModule, Trainer
from lightning.pytorch.callbacks import BackboneFinetuning
from lightning.pytorch.loggers import WandbLogger
from transformers import ViTModel
from lightning.pytorch.callbacks import early_stopping
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

In [ ]:
## loading the data and directory handling
! git clone https://git.wur.nl/lobry001/ucmdata.git
os.chdir('ucmdata')

with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
  zip_ref.extractall('UCMImages')

!mv UCMImages/UCMerced_LandUse/Images .
!rm -rf UCMImages README.md UCMerced_LandUse.zip
!ls

UCM_images_path = "Images/"
Multilabels_path = "LandUse_Multilabeled.txt"

In [ ]:
## Accessing the images
dir = "/content/ucmdata/Images"
image_dict = {} # dict for key:class, value: [image_paths]
labels_map = {} # dict for key:class_num, value: class_name

for class_num, class_name in enumerate(os.listdir(dir)):
  labels_map[class_num] = class_name
  class_dir = os.path.join(dir, class_name)

  if os.path.isdir(class_dir): # sort out *.txt-file
    image_paths = [os.path.join(class_dir, image) for image in os.listdir(class_dir)]
    image_dict[class_num] = image_paths


In [ ]:
def split_train_test(image_dict, val_ratio, test_ratio):
   """
    Splits image paths into training, validation, and test datasets.

    Args:
        image_dict: A dictionary where keys are class labels and values are lists of image paths belonging to that class.
        val_ratio: The ratio of images to allocate for the validation set.
        test_ratio: The ratio of images to allocate for the test set.

    Returns:
        A tuple containing three lists: (train_dataset, val_dataset, test_dataset), each list contains tuples of (image_path, label).
    """

  random.seed(42) # to ensure the same test set for different models
  train_dataset = []
  val_dataset = []
  test_dataset = []

  for label, image_paths in image_dict.items():

    sample_length = len(image_paths)

    ## split off the test_dataset with the fixed random seed first to keep test set consistent
    random.shuffle(image_paths)

    test_split = int(sample_length * test_ratio)
    test_dataset.extend([(image, label) for image in image_paths[:test_split]])
    image_paths = image_paths[test_split:]

    ## reshuffle to mix up training and validation data for different runs
    random.seed()
    random.shuffle(image_paths)

    val_split = int(sample_length * val_ratio)

    val_dataset.extend([(image, label) for image in image_paths[:val_split]])
    train_dataset.extend([(image, label) for image in image_paths[val_split:]])

  return train_dataset, val_dataset, test_dataset

train_dataset,val_dataset, test_dataset = split_train_test(image_dict, val_ratio = 0.15, test_ratio=0.15)

In [ ]:
def plot_first_per_class(data, index_to_class, cols=5, figsize=(15, 3)):
    """
    Plots the first image for each class in the given dataset.

    Args:
        data: A list of (image_path, label) tuples.
        index_to_class: A dictionary mapping numerical labels to class names.
        cols: Number of columns in the plot grid. Defaults to 5.
        figsize: Figure size for the plot. Defaults to (15, 3).
    """

    # Collect first image per label
    first_image_per_label = {}
    for image_path, label in data:
        if label not in first_image_per_label:
            first_image_per_label[label] = image_path

    labels = sorted(first_image_per_label.keys())
    num_classes = len(labels)
    rows = (num_classes + cols - 1) // cols

    plt.figure(figsize=(figsize[0], figsize[1] * rows))

    for i, label in enumerate(labels):
        image_path = first_image_per_label[label]
        class_name = index_to_class.get(label, "unknown")

        with Image.open(image_path) as img:
            img = img.convert("RGB")

            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.title(f"{label}: {class_name}")
            plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_first_per_class(test_dataset, labels_map) ## should plot the same images in every project!!

In [ ]:
class UCMDataset(Dataset):
  """
  Custom Dataset class for the UCMerced Land Use dataset which loads images from file paths,
  applies transformations, returns them as tensors usable for training.

  """
#Custom Dataset for our images
  def __init__(self, samples_list, transform=None):
      """Initializes the UCMDataset."""

    self.samples = samples_list
    self.transform = transform

  def __len__(self):
 """Returns the total number of samples in the dataset, used for bacthing and iteration."""
    return len(self.samples)

  def __getitem__(self, index):
     """
    Retrieves an image and its corresponding label by index.

    Args:
        index: The index of the sample to retrieve.

    Returns:
        A tuple containing the image and its label.
    """
    img_path = self.samples[index][0]
    image = Image.open(img_path)
    label = self.samples[index][1]
    if self.transform:
      image = self.transform(image)
    return image, label

In [ ]:
## define the transforms
transforms_train = v2.Compose([
    v2.Resize((224, 224)), ## valid resnet image size, fix passing size as tuple
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.CenterCrop((224, 224)),
    v2.RandomRotation(degrees=(-10, 10)),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean= [0.485, 0.456,0.406],
                 std=[0.229, 0.224, 0.225])  ## imagenet mean and std
])

transforms_test = v2.Compose([
        v2.Resize((224, 224)), ## valid resnet image size, fix passing size as tuple
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean= [0.485, 0.456,0.406],
                     std=[0.229, 0.224, 0.225])  ## imagenet mean and std
])

train_ds = UCMDataset(train_dataset, transform=transforms_train)
val_ds = UCMDataset(val_dataset, transform=transforms_test)
test_ds = UCMDataset(test_dataset, transform=transforms_test)

train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")

# Define mean and std for unnormalization (from the transforms used)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

img = train_features[0].squeeze()
label = train_labels[0]

# Unnormalize the image
img = img * std + mean
# Clamp values to [0, 1] range to ensure valid display for imshow
img = torch.clamp(img, 0, 1)

# Permute the dimensions from (C, H, W) to (H, W, C) for matplotlib
plt.imshow(img.permute(1, 2, 0))
plt.show()
print(f"Label: {labels_map[label.item()]}")

# 1. Fine-tuned Resnet18 Model, vision transformer


In [ ]:
# setting up weights and biases
import wandb
wandb.login(relogin =True)

In [ ]:
class ViTClassifier(LightningModule):
    """
    A Vision Transformer (ViT) based classifier using PyTorch Lightning.

    This model utilizes a pre-trained ViT backbone and adds a custom classification head.
    It supports training, validation, testing, and prediction steps.

    Args:
        num_classes: The number of output classes for classification.
        learning_rate: The learning rate for the optimizer.
    """
    def __init__(self, num_classes=21, learning_rate=1e-3):
    """
        Initializes the Transformer based classifier.

        Args:
            num_classes: The number of output classes for the classifier. Defaults to 21.
            learning_rate: The learning rate for the optimizer.
            ##freeze: If True, freezes the backbone layers and only trains the head. Defaults to True.
            ##dropout: If True, adds a dropout layer to the classification head. Defaults to True.
        """
        super().__init__()
        self.save_hyperparameters()

        # Create backbone from pretrained ViT
        self.backbone = ViTModel.from_pretrained("google/vit-base-patch16-224")

        # Add custom classification head
        # The hidden size for 'vit-base-patch16-224' is 768
        self.head = nn.Sequential(
            nn.LayerNorm(self.backbone.config.hidden_size), #normalization
            nn.Linear(self.backbone.config.hidden_size, 512),
            # no flatten, ViT output is already 1D
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        """
        Performs a forward pass through the model.

        Args:
            x: The input tensor (image batch).

        Returns:
            torch.Tensor: The output logits from the classification head.
        """
        # Extract features with backbone
        outputs = self.backbone(x)
        features = outputs.last_hidden_state[:, 0, :]
        # Classify with head
        return self.head(features)

    def training_step(self, batch, batch_idx):
        """
        Performs a single training step.

        Args:
            batch: A tuple containing the input images and their labels.
            batch_idx: The index of the current batch.

        Returns:
            torch.Tensor: The calculated loss for the training step.
        """
        x, y = batch
        y_hat = self(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss)
        self.log('train_acc', acc)
        return loss

    def validation_step(self, batch, batch_idx):
        """
        Performs a single validation step.

        Args:
            batch: A tuple containing the input images and their labels.
            batch_idx: The index of the current batch.

        Returns:
            torch.Tensor: The calculated loss for the validation step.
        """
        x, y = batch
        y_hat = self(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('val_loss', loss)
        self.log('val_acc', acc)
        return loss

    def test_step(self, batch, batch_idx):
        """
        Performs a single test step.

        Args:
            batch: A tuple containing the input images and their labels.
            batch_idx: The index of the current batch.

        Returns:
            torch.Tensor: The calculated loss for the test step.
        """
        x, y = batch
        y_hat = self(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('test_loss', loss)
        self.log('test_acc', acc)
        return loss

    def predict_step(self, batch, batch_idx):
        """
        Performs a single prediction step.

        Args:
            batch: A tuple containing the input images and their labels.
            batch_idx: The index of the current batch.

        Returns:
            A tuple containing the predicted labels and the true labels.
        """
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        return preds, y

    def configure_optimizers(self):
        """
        Configures the optimizer for the model.

        Returns:
            torch.optim.Optimizer: The Adam optimizer configured for the model.
        """
        # Initially, only optimize the parameters of the head.
        # The BackboneFinetuning callback will handle adding and unfreezing the backbone parameters later.
        return torch.optim.Adam(self.head.parameters(), lr=self.hparams.learning_rate)

In [ ]:
# Close any previous run from the notebook session
if wandb.run is not None:
    wandb.finish()

# Create a brand-new W&B run
wandb_logger = WandbLogger(
    project="resnet50",
    name=None,          # optional: let W&B auto-name it
    #log_model=True     # optional
)

wandb_logger.experiment.config.update({
    "Dataset": "UCM single label",
    "Augmentation": "Yes - Fixed for only Train - Added CenterCrop",
    "Early Stopping": "Yes at 3",
    "Epochs": 15,
    "Model": "Vision Transformer"
})

# Define the BackboneFinetuning callback
backbone_finetuning = BackboneFinetuning(
    unfreeze_backbone_at_epoch=5,  # Start unfreezing backbone earlier, e.g., at epoch 5
    lambda_func=lambda epoch: 1.5,  # Gradually increase backbone learning rate
    backbone_initial_ratio_lr=0.01,  # Backbone starts at 1% of head learning rate
    should_align=True,  # Align rates when backbone rate reaches head rate
    verbose=True,  # Print learning rates during training
)

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min") # mode=min says that the lowest for val_loss is the better

model = ResNetClassifier()

trainer = Trainer(
    callbacks=[backbone_finetuning, early_stopping],
    max_epochs=15,
    logger=wandb_logger
)

trainer.fit(model, train_dataloader, val_dataloader)

# Mark this run finished so the next rerun starts a new one
#wandb.finish()

In [ ]:
# use seafile for confusion matrix
#trainer.test(dataloaders=test_dataloader)
preds = trainer.predict(model, val_dataloader)

all_preds = torch.cat([p[0] for p in preds])
all_targets = torch.cat([p[1] for p in preds])




In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_targets, all_preds)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_map.values(),
    yticklabels=labels_map.values())
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()



In [ ]:
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=all_targets.numpy(),
        preds=all_preds.numpy(),
        class_names= list(labels_map.values())
    )
})

## 2. Look at mismatches

In [ ]:
import torch

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

misclassified = []

with torch.no_grad():
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        for i in range(len(x)):
            if preds[i] != y[i]:
                misclassified.append((
                    x[i].cpu(),         # image
                    preds[i].cpu(),     # predicted
                    y[i].cpu()          # true
                ))

In [ ]:

num_samples = min(10, len(misclassified))
samples = random.sample(misclassified, num_samples)

In [ ]:

class_names = [labels_map[i] for i in range(len(labels_map))]

plt.figure(figsize=(15, 6))

for i, (img, pred, true) in enumerate(samples):
    plt.subplot(2, 5, i + 1)

    # If tensor is (C, H, W) → convert to (H, W, C)
    img = img.permute(1, 2, 0)

    mean = torch.tensor([0.485, 0.456, 0.406])
    std = torch.tensor([0.229, 0.224, 0.225])

    img = img * std + mean
    img = img.clamp(0, 1)
    # If normalized → you may need to unnormalize here
    plt.imshow(img)
    plt.axis("off")

    plt.title(f"T: {class_names[true]}\nP: {class_names[pred]}")

plt.tight_layout()
plt.show()